# 5-way classification: inference and evaluation

## 1. Setup

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

TEST_DF_PATH = Path("predictions/test_df.csv")
REGISTRY_PATH = Path("roberta_models/best_models_registry.json")
MODELS_DIR = Path("roberta_models")

TEXT_COL = "sent_text"
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

Device: cuda


## 2. Load test data

In [2]:
test_df = pd.read_csv(TEST_DF_PATH)
test_df["label"] = test_df["label"].fillna("None")

print("Rows:", len(test_df))
test_df.head()

Rows: 1502


,case_name,sent_text,label,hdr_title,hdr_match,hdr_group,y_labelled
0,ECLI:NL:GHAMS:2015:2960.txt,De kantonrechter heeft in het bestreden vonnis...,None,Feiten,1,Feiten,0
1,ECLI:NL:GHAMS:2015:2960.txt,Deze feiten zijn in hoger beroep niet in gesch...,None,Feiten,1,Feiten,0
2,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [geïntimeerde] een bedrag van...,materiele feiten,Beoordeling,1,Beoordeling,1
3,ECLI:NL:GHAMS:2015:2960.txt,Op [datum] heeft [appellante] een schriftelijk...,materiele feiten,Beoordeling,1,Beoordeling,1
4,ECLI:NL:GHAMS:2015:2960.txt,Deze verklaring houdt onder meer in: “Dit bedr...,None,Beoordeling,1,Beoordeling,0


## 3. Load the 5-way model

Resolved from `best_models_registry.json` under the `text_only_5way` key, rather than a hardcoded path — so this always points at whichever checkpoint the registry currently designates as best.

In [3]:
with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
    registry = json.load(f)

model_info = registry["text_only_5way"]
model_path = MODELS_DIR / Path(model_info["saved_path"]).name
labels = model_info["labels_order"]
max_len = model_info.get("max_len", 256)

if not model_path.exists():
    raise FileNotFoundError(f"Model path not found: {model_path}")

print("Model path:", model_path)
print("Labels (in order):", labels)

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to(DEVICE)
model.eval()

Model path: roberta_models\text_only_5way_best_20260203_101227
Labels (in order): ['None', 'beoordeling', 'beslissing', 'materiele feiten', 'proceshandelingen']


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(40000, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

## 4. Run inference

Batched forward pass over the test set, softmax over logits to get calibrated-looking class probabilities.

In [4]:
@torch.no_grad()
def predict_probs(texts, tokenizer, model, max_len, batch_size=32):
    all_probs = []

    for start in tqdm(range(0, len(texts), batch_size), desc="5-way inference"):
        batch = texts[start:start + batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        all_probs.append(probs.cpu().numpy())

    return np.vstack(all_probs)


texts = test_df[TEXT_COL].fillna("").astype(str).tolist()
probs = predict_probs(texts, tokenizer, model, max_len, BATCH_SIZE)

pred_idx = probs.argmax(axis=1)
pred_label = np.array([labels[i] for i in pred_idx])

probs.shape

5-way inference:   0%|          | 0/47 [00:00<?, ?it/s]

(1502, 5)

## 5. Attach predictions and save

In [5]:
for i, label in enumerate(labels):
    col = "text5_p_" + label.lower().replace(" ", "_")
    test_df[col] = probs[:, i]

test_df["text5_pred_idx"] = pred_idx
test_df["text5_pred_label"] = pred_label

OUTPUT_PATH = Path("predictions/five_way_inference.csv")
test_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT_PATH)
test_df[["sent_text", "label", "text5_pred_label"]].head()

Saved: predictions\five_way_inference.csv


,sent_text,label,text5_pred_label
0,De kantonrechter heeft in het bestreden vonnis...,None,beoordeling
1,Deze feiten zijn in hoger beroep niet in gesch...,None,beoordeling
2,Op [datum] heeft [geïntimeerde] een bedrag van...,materiele feiten,materiele feiten
3,Op [datum] heeft [appellante] een schriftelijk...,materiele feiten,materiele feiten
4,Deze verklaring houdt onder meer in: “Dit bedr...,None,None


> **Landmine when reloading this CSV elsewhere:** the `label` and `text5_pred_label` columns contain the literal string `"None"`. `pandas.read_csv` treats `"None"` as a missing value by default and silently turns it into `NaN`, which breaks any `==`/`!=` comparison between the two columns (`NaN != NaN` is always `True`). Any notebook that reloads `predictions/five_way_inference.csv` must pass `keep_default_na=False` to `read_csv`, exactly as the existing `new_resutls.ipynb` already does for the same reason.

## 6. Evaluate

Sanity check against gold `label`. This should reproduce the previously reported 5-way numbers (accuracy 0.6658, macro F1 0.6680) if the same checkpoint and test set are being used — if it doesn't match, something changed (different checkpoint, different test split, or a preprocessing difference) and is worth chasing down before moving to the next formulation.

In [6]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

LABELS_5WAY = ["None", "beoordeling", "beslissing", "materiele feiten", "proceshandelingen"]

y_true = test_df["label"]
y_pred = test_df["text5_pred_label"]

print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
print()
print(classification_report(y_true, y_pred, labels=LABELS_5WAY, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=LABELS_5WAY)
pd.DataFrame(
    cm,
    index=[f"true_{l}" for l in LABELS_5WAY],
    columns=[f"pred_{l}" for l in LABELS_5WAY]
)

Accuracy: 0.6658

                   precision    recall  f1-score   support

             None     0.7033    0.7579    0.7295       541
      beoordeling     0.6355    0.6992    0.6659       389
       beslissing     0.8947    0.7391    0.8095        69
 materiele feiten     0.6423    0.5320    0.5820       297
proceshandelingen     0.5798    0.5291    0.5533       206

         accuracy                         0.6658      1502
        macro avg     0.6911    0.6515    0.6680      1502
     weighted avg     0.6655    0.6658    0.6634      1502



,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,410,75,4,31,21
true_beoordeling,59,272,2,31,25
true_beslissing,14,2,51,2,0
true_materiele feiten,76,30,0,158,33
true_proceshandelingen,24,49,0,24,109


## 7. Error analysis

Everything below works off the predictions already attached to `test_df` (`text5_pred_label` and the per-class probability columns) — no new inference.

### 7.1 Confusion pairs

Every (true label → predicted label) combination, excluding correct predictions, ranked by how often it happens.

In [7]:
LABEL_PROB_COLS = {
    "None": "text5_p_none",
    "beoordeling": "text5_p_beoordeling",
    "beslissing": "text5_p_beslissing",
    "materiele feiten": "text5_p_materiele_feiten",
    "proceshandelingen": "text5_p_proceshandelingen",
}

is_error = test_df["label"] != test_df["text5_pred_label"]
errors_df = test_df[is_error].copy()

print(f"Total errors: {len(errors_df)} / {len(test_df)} ({100 * len(errors_df) / len(test_df):.2f}%)")

support_by_label = test_df["label"].value_counts()

confusion_pairs = (
    errors_df.groupby(["label", "text5_pred_label"])
    .size()
    .rename("count")
    .reset_index()
)
confusion_pairs["pct_of_true_label_support"] = confusion_pairs.apply(
    lambda r: round(100 * r["count"] / support_by_label[r["label"]], 2),
    axis=1
)
confusion_pairs = confusion_pairs.sort_values("count", ascending=False).reset_index(drop=True)

confusion_pairs

Total errors: 502 / 1502 (33.42%)


,label,text5_pred_label,count,pct_of_true_label_support
0,materiele feiten,None,76,25.59
1,None,beoordeling,75,13.86
2,beoordeling,None,59,15.17
3,proceshandelingen,beoordeling,49,23.79
4,materiele feiten,proceshandelingen,33,11.11
5,None,materiele feiten,31,5.73
6,beoordeling,materiele feiten,31,7.97
7,materiele feiten,beoordeling,30,10.10
8,beoordeling,proceshandelingen,25,6.43
9,proceshandelingen,None,24,11.65


### 7.2 Relevance errors vs. role-confusion errors

Splits every mistake into two kinds: the model got the relevance boundary wrong (predicted None for a real role, or predicted a role for a None sentence), vs. the model correctly judged the sentence relevant but picked the wrong one of the four roles.

In [8]:
def classify_error(row):
    if row["label"] == row["text5_pred_label"]:
        return "correct"
    if row["label"] == "None" or row["text5_pred_label"] == "None":
        return "relevance_error"
    return "role_confusion_error"

test_df["error_type"] = test_df.apply(classify_error, axis=1)

overall_counts = test_df["error_type"].value_counts()
overall_pct = (100 * overall_counts / len(test_df)).round(2)

print("Share of all test sentences:")
display(pd.DataFrame({"count": overall_counts, "pct_of_all_sentences": overall_pct}))

errors_only_counts = errors_df.assign(
    error_type=lambda d: d.apply(classify_error, axis=1)
)["error_type"].value_counts()
errors_only_pct = (100 * errors_only_counts / len(errors_df)).round(2)

print("\nShare among errors only:")
display(pd.DataFrame({"count": errors_only_counts, "pct_of_all_errors": errors_only_pct}))

Share of all test sentences:


,count,pct_of_all_sentences
error_type,,
correct,1000,66.58
relevance_error,304,20.24
role_confusion_error,198,13.18



Share among errors only:


,count,pct_of_all_errors
error_type,,
relevance_error,304,60.56
role_confusion_error,198,39.44


### 7.3 Per-class destination of errors (recall-normalized confusion matrix)

For each gold label, where do its sentences end up, as a percentage. Diagonal = recall.

In [9]:
recall_matrix = pd.crosstab(
    test_df["label"],
    test_df["text5_pred_label"],
    normalize="index"
) * 100

recall_matrix = recall_matrix.reindex(
    index=LABELS_5WAY,
    columns=LABELS_5WAY,
    fill_value=0
).round(2)

recall_matrix

text5_pred_label,None,beoordeling,beslissing,materiele feiten,proceshandelingen
label,,,,,
None,75.79,13.86,0.74,5.73,3.88
beoordeling,15.17,69.92,0.51,7.97,6.43
beslissing,20.29,2.90,73.91,2.90,0.00
materiele feiten,25.59,10.10,0.00,53.20,11.11
proceshandelingen,11.65,23.79,0.00,11.65,52.91


### 7.4 Model confidence: correct vs. incorrect predictions

`pred_confidence` is the probability the model assigned to whichever label it actually predicted. If wrong predictions cluster at low confidence, the model is at least "aware" it's on shaky ground. If wrong predictions are high-confidence, that's a more concerning failure mode — confidently wrong, not just borderline.

In [10]:
test_df["pred_confidence"] = test_df[list(LABEL_PROB_COLS.values())].max(axis=1)

is_correct = test_df["label"] == test_df["text5_pred_label"]

confidence_summary = (
    test_df.groupby(is_correct)["pred_confidence"]
    .describe()[["count", "mean", "50%", "std", "min", "max"]]
)
confidence_summary.index = ["incorrect", "correct"]
display(confidence_summary.round(3))

HIGH_CONF_THRESHOLD = 0.8
high_conf_wrong = test_df[(~is_correct) & (test_df["pred_confidence"] > HIGH_CONF_THRESHOLD)]

print(
    f"\nHigh-confidence (>{HIGH_CONF_THRESHOLD}) wrong predictions: "
    f"{len(high_conf_wrong)} / {(~is_correct).sum()} errors "
    f"({100 * len(high_conf_wrong) / (~is_correct).sum():.2f}% of all errors)"
)

,count,mean,50%,std,min,max
incorrect,502.0,0.743,0.763,0.170,0.300,0.988
correct,1000.0,0.857,0.919,0.148,0.341,0.990



High-confidence (>0.8) wrong predictions: 224 / 502 errors (44.62% of all errors)


### 7.5 Where errors cluster structurally

Error rate by normalized section-header group, and a check for whether mistakes are spread evenly across the 100 source decisions or concentrated in a handful of documents.

In [11]:
hdr_error_rate = (
    test_df
    .assign(is_error=~is_correct)
    .groupby("hdr_group")["is_error"]
    .agg(n_sentences="count", n_errors="sum")
)
hdr_error_rate["error_rate_pct"] = (100 * hdr_error_rate["n_errors"] / hdr_error_rate["n_sentences"]).round(2)
hdr_error_rate = hdr_error_rate.sort_values("error_rate_pct", ascending=False)

hdr_error_rate

,n_sentences,n_errors,error_rate_pct
hdr_group,,,
Proceshandelingen partijen,252,97,38.49
Beoordeling,676,246,36.39
Beslissing,237,68,28.69
Feiten,288,78,27.08
Context,49,13,26.53


In [12]:
case_error_counts = (
    test_df
    .assign(is_error=~is_correct)
    .groupby("case_name")["is_error"]
    .agg(n_sentences="count", n_errors="sum")
)
case_error_counts["error_rate_pct"] = (
    100 * case_error_counts["n_errors"] / case_error_counts["n_sentences"]
).round(2)

print(f"Test-set documents: {len(case_error_counts)}")
print(f"Documents with zero errors: {(case_error_counts['n_errors'] == 0).sum()}")

sorted_by_errors = case_error_counts.sort_values("n_errors", ascending=False)
total_errors = sorted_by_errors["n_errors"].sum()
cum_pct = 100 * sorted_by_errors["n_errors"].cumsum() / total_errors

print("\nCumulative share of all errors covered by the top-N error-heaviest documents:")
for n in [5, 10, 20]:
    print(f"  top {n} documents: {cum_pct.iloc[n - 1]:.1f}%")

print("\nTop 10 documents by error count:")
sorted_by_errors.head(10)

Test-set documents: 20
Documents with zero errors: 2

Cumulative share of all errors covered by the top-N error-heaviest documents:
  top 5 documents: 52.2%
  top 10 documents: 81.9%
  top 20 documents: 100.0%

Top 10 documents by error count:


,n_sentences,n_errors,error_rate_pct
case_name,,,
ECLI:NL:RBOBR:2020:3584.txt,221,69,31.22
ECLI:NL:RBOVE:2017:1505.txt,129,55,42.64
ECLI:NL:RBDHA:2018:3316.txt,121,51,42.15
ECLI:NL:RBNNE:2015:2927.txt,99,44,44.44
ECLI:NL:HR:2018:874.txt,108,43,39.81
ECLI:NL:RBOBR:2016:6963.txt,123,38,30.89
ECLI:NL:RBNNE:2016:4308.txt,77,35,45.45
ECLI:NL:RBDHA:2021:6182.txt,76,27,35.53
ECLI:NL:GHARL:2017:8427.txt,148,26,17.57


### 7.6 Qualitative spot-check

Sample sentences from the top confusion pairs (from 7.1), with the model's confidence and document/header context, to read and judge whether each is a genuine model failure or a borderline/context-dependent case.

In [13]:
N_TOP_PAIRS = 5
N_EXAMPLES_PER_PAIR = 5

top_pairs = confusion_pairs.head(N_TOP_PAIRS)[["label", "text5_pred_label"]].itertuples(index=False)

for true_label, pred_label in top_pairs:
    subset = test_df[
        (test_df["label"] == true_label) & (test_df["text5_pred_label"] == pred_label)
    ]

    print("=" * 90)
    print(f"True: {true_label}  ->  Predicted: {pred_label}   ({len(subset)} cases total)")
    print("=" * 90)

    sample = subset.sample(min(N_EXAMPLES_PER_PAIR, len(subset)), random_state=42)
    for _, row in sample.iterrows():
        print(f"- [{row['case_name']} | hdr={row['hdr_group']} | conf={row['pred_confidence']:.2f}] {row['sent_text']}")
    print()

True: materiele feiten  ->  Predicted: None   (76 cases total)
- [ECLI:NL:RBOBR:2020:3584.txt | hdr=Feiten | conf=0.49] In een memo van diezelfde datum gericht aan de Raad van Bestuur bericht [naam hoofd BDO Advisory] (hoofd BDO Advisory en lid van de Executive Board van BDO Holding) over de stand van zaken inzake de juridische procedure rondom de BTW-kwestie.
- [ECLI:NL:RBNNE:2015:2927.txt | hdr=Beoordeling | conf=0.96] Pas na de provocaties vanuit de groep van verdachte kwam het tot een confrontatie in de [straat 2].
- [ECLI:NL:RBNNE:2015:2927.txt | hdr=Beoordeling | conf=0.97] Verdachte, [medeverdachte 1] en [medeverdachte 2] besloten niet meer naar [bedrijfsnaam] te gaan en ze staken [straat 1] over in de richting van de [straat 2].
- [ECLI:NL:GHAMS:2015:2960.txt | hdr=Beoordeling | conf=0.52] Uit de overeenkomst tussen partijen volgt dat [appellante] het geleende bedrag zal terugbetalen wanneer zij daartoe in staat zal zijn.
- [ECLI:NL:RBOVE:2017:1505.txt | hdr=Feiten | conf=0.96]

## 8. Header-prior Bayesian fusion

Combines the model's text-based probabilities with a prior estimated from each sentence's section-header group (`hdr_group`), learned from the training set:

```
log P(y | s, h) ∝ log P(y | s) + λ · log P(y | h)
```

Reports two versions of λ:
- **λ = 1** — matches what the paper's methods section currently states ("equal weight").
- **tuned λ** — swept on the 1281-row validation set (not the test set) over a small grid, picking whichever maximizes macro F1, then applied once to the test set. This replaces guessing with evidence, and resolves the earlier-found mismatch between the paper's stated λ=1 and the original code's actual λ=3.

In [14]:
TRAIN_DF_PATH = Path("predictions/train_df.csv")
EVAL_DF_PATH = Path("predictions/eval_df.csv")
ALPHA = 1.0

train_df = pd.read_csv(TRAIN_DF_PATH, keep_default_na=False)
eval_df = pd.read_csv(EVAL_DF_PATH, keep_default_na=False)

print("Train rows:", len(train_df))
print("Validation rows:", len(eval_df))


def make_header_prior(df_train, label_col, labels, alpha=1.0):
    counts = (
        df_train.groupby(["hdr_group", label_col]).size()
        .unstack(fill_value=0)
        .reindex(columns=labels, fill_value=0)
    )
    probs = counts + alpha
    return probs.div(probs.sum(axis=1), axis=0)


def make_global_prior(df_train, label_col, labels, alpha=1.0):
    counts = df_train[label_col].value_counts().reindex(labels, fill_value=0) + alpha
    return (counts / counts.sum()).values


def get_meta_prior_df(df_apply, header_prior_train, global_prior, labels):
    def get_prior(hdr_group):
        if hdr_group in header_prior_train.index:
            return header_prior_train.loc[hdr_group].values
        return global_prior

    mat = np.vstack(df_apply["hdr_group"].apply(get_prior))
    return pd.DataFrame(mat, columns=labels, index=df_apply.index)


def combine_log_scores(text_probs, metadata_probs, lam, eps=1e-12):
    text = np.clip(text_probs.values, eps, 1.0)
    metadata = np.clip(metadata_probs.values, eps, 1.0)

    scores = np.log(text) + lam * np.log(metadata)
    scores -= scores.max(axis=1, keepdims=True)
    scores = np.exp(scores)
    scores /= scores.sum(axis=1, keepdims=True)

    return pd.DataFrame(scores, columns=text_probs.columns, index=text_probs.index)

Train rows: 5608
Validation rows: 1281


### 8.1 Run the 5-way model on the validation set (needed to tune λ)

In [15]:
eval_texts = eval_df[TEXT_COL].fillna("").astype(str).tolist()
eval_probs = predict_probs(eval_texts, tokenizer, model, max_len, BATCH_SIZE)

eval_text_probs = pd.DataFrame(eval_probs, columns=labels, index=eval_df.index)[LABELS_5WAY]

print("Eval text-probability matrix shape:", eval_text_probs.shape)

5-way inference:   0%|          | 0/41 [00:00<?, ?it/s]

Eval text-probability matrix shape: (1281, 5)


### 8.2 Build the header prior and tune λ on validation

In [16]:
from sklearn.metrics import precision_recall_fscore_support

header_prior_5way = make_header_prior(train_df, "label", LABELS_5WAY, ALPHA)
global_prior_5way = make_global_prior(train_df, "label", LABELS_5WAY, ALPHA)

eval_meta_prior = get_meta_prior_df(eval_df, header_prior_5way, global_prior_5way, LABELS_5WAY)

LAMBDA_GRID = [0, 0.1, 0.25, 0.5, 1, 2, 3, 5]
sweep_results = []

for lam in LAMBDA_GRID:
    fused = combine_log_scores(eval_text_probs, eval_meta_prior, lam)
    pred = fused.idxmax(axis=1)
    _, _, f1, _ = precision_recall_fscore_support(
        eval_df["label"], pred, labels=LABELS_5WAY, average="macro", zero_division=0
    )
    sweep_results.append({"lambda": lam, "val_macro_f1": round(f1, 4)})

sweep_df = pd.DataFrame(sweep_results)
best_lambda = sweep_df.loc[sweep_df["val_macro_f1"].idxmax(), "lambda"]

print("Validation lambda sweep:")
print(sweep_df)
print(f"\nBest lambda on validation: {best_lambda}")

Validation lambda sweep:
   lambda  val_macro_f1
0    0.00        0.6956
1    0.10        0.6962
2    0.25        0.6981
3    0.50        0.7071
4    1.00        0.6603
5    2.00        0.6528
6    3.00        0.6357
7    5.00        0.4715

Best lambda on validation: 0.5


### 8.3 Apply to the test set: λ=1 vs. tuned λ vs. no fusion

In [17]:
test_prob_cols = {l: "text5_p_" + l.lower().replace(" ", "_") for l in LABELS_5WAY}
test_text_probs = test_df[[test_prob_cols[l] for l in LABELS_5WAY]].copy()
test_text_probs.columns = LABELS_5WAY

test_meta_prior = get_meta_prior_df(test_df, header_prior_5way, global_prior_5way, LABELS_5WAY)

results = []

for lam, name in [(0, "no fusion (lambda=0)"), (1, "lambda = 1"), (best_lambda, f"tuned lambda ({best_lambda})")]:
    fused_test = combine_log_scores(test_text_probs, test_meta_prior, lam)
    pred_test = fused_test.idxmax(axis=1)

    acc = accuracy_score(test_df["label"], pred_test)
    _, _, f1, _ = precision_recall_fscore_support(
        test_df["label"], pred_test, labels=LABELS_5WAY, average="macro", zero_division=0
    )
    results.append({"setting": name, "lambda": lam, "test_accuracy": round(acc, 4), "test_macro_f1": round(f1, 4)})
    test_df[f"fivewayheader_pred_lam_{lam}"] = pred_test

results_df = pd.DataFrame(results)
print(results_df)

                setting  lambda  test_accuracy  test_macro_f1
0  no fusion (lambda=0)     0.0         0.6658         0.6680
1            lambda = 1     1.0         0.6771         0.6766
2    tuned lambda (0.5)     0.5         0.6758         0.6772


In [18]:
tuned_pred_col = f"fivewayheader_pred_lam_{best_lambda}"
print(f"Full classification report at tuned lambda={best_lambda} (test set)")
print(classification_report(test_df["label"], test_df[tuned_pred_col], labels=LABELS_5WAY, digits=4, zero_division=0))

Full classification report at tuned lambda=0.5 (test set)
                   precision    recall  f1-score   support

             None     0.6933    0.7689    0.7292       541
      beoordeling     0.6548    0.7069    0.6799       389
       beslissing     0.9074    0.7101    0.7967        69
 materiele feiten     0.6639    0.5387    0.5948       297
proceshandelingen     0.6150    0.5583    0.5852       206

         accuracy                         0.6758      1502
        macro avg     0.7069    0.6566    0.6772      1502
     weighted avg     0.6766    0.6758    0.6732      1502



## 9. Confidence thresholding (None-first decision rule)

Replaces plain argmax-over-5 with a two-step rule: if `P(None) ≥ θ_none`, predict None; otherwise predict the argmax of the 4 role probabilities (still mutually exclusive, so there's one clear winner once None is ruled out).

`θ_none` is estimated by running the model on the **training set** (5608 sentences, gold labels available) and sweeping candidate thresholds — but instead of maximizing overall macro F1 of the final decision rule, it maximizes the **None class's own binary F1** (treating it as "is this sentence None, yes or no" in isolation, independent of how well the other 4 classes come out). The precision-maximizing threshold is reported alongside for comparison. Validation-set performance at the chosen threshold is reported as a transparency check.

In [19]:
LEGAL_LABELS = [l for l in LABELS_5WAY if l != "None"]

train_texts = train_df[TEXT_COL].fillna("").astype(str).tolist()
train_probs = predict_probs(train_texts, tokenizer, model, max_len, BATCH_SIZE)
train_text_probs = pd.DataFrame(train_probs, columns=labels, index=train_df.index)[LABELS_5WAY]

print("Train text-probability matrix shape:", train_text_probs.shape)


def none_first_decision(text_probs, theta_none):
    p_none = text_probs["None"]
    role_argmax = text_probs[LEGAL_LABELS].idxmax(axis=1)
    return pd.Series(np.where(p_none >= theta_none, "None", role_argmax), index=text_probs.index)


train_is_none = (train_df["label"] == "None").astype(int)

THETA_GRID = np.round(np.arange(0.05, 1.0, 0.05), 2)
sweep_results = []

for theta in THETA_GRID:
    pred_is_none = (train_text_probs["None"] >= theta).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        train_is_none, pred_is_none, labels=[0, 1], average="binary", pos_label=1, zero_division=0
    )
    sweep_results.append({
        "theta_none": theta,
        "none_precision": round(precision, 4),
        "none_recall": round(recall, 4),
        "none_f1": round(f1, 4)
    })

sweep_df = pd.DataFrame(sweep_results)
best_theta_f1 = sweep_df.loc[sweep_df["none_f1"].idxmax(), "theta_none"]
best_theta_precision = sweep_df.loc[sweep_df["none_precision"].idxmax(), "theta_none"]
best_theta_none = best_theta_f1

print(sweep_df.to_string(index=False))
print(f"\nBest theta_none maximizing None-class F1: {best_theta_f1}")
print(f"Best theta_none maximizing None-class precision: {best_theta_precision}")
print(f"Using F1-maximizing threshold going forward: {best_theta_none}")

5-way inference:   0%|          | 0/176 [00:00<?, ?it/s]

Train text-probability matrix shape: (5608, 5)
 theta_none  none_precision  none_recall  none_f1
       0.05          0.5880       0.9855   0.7365
       0.10          0.7174       0.9725   0.8257
       0.15          0.7857       0.9604   0.8644
       0.20          0.8222       0.9449   0.8793
       0.25          0.8530       0.9324   0.8909
       0.30          0.8678       0.9239   0.8950
       0.35          0.8885       0.9179   0.9030
       0.40          0.9005       0.9064   0.9034
       0.45          0.9130       0.8988   0.9059
       0.50          0.9236       0.8898   0.9064
       0.55          0.9329       0.8843   0.9080
       0.60          0.9399       0.8688   0.9029
       0.65          0.9479       0.8563   0.8998
       0.70          0.9572       0.8408   0.8952
       0.75          0.9632       0.8247   0.8886
       0.80          0.9692       0.8027   0.8781
       0.85          0.9734       0.7702   0.8599
       0.90          0.9781       0.7166   0.8272
   

### 9.1 Apply the tuned threshold to validation (transparency check) and test

In [20]:
val_pred_threshold = none_first_decision(eval_text_probs, best_theta_none)
_, _, val_f1, _ = precision_recall_fscore_support(
    eval_df["label"], val_pred_threshold, labels=LABELS_5WAY, average="macro", zero_division=0
)
print(f"[Transparency check] Validation macro F1 at theta_none={best_theta_none}: {val_f1:.4f}")

test_pred_threshold = none_first_decision(test_text_probs, best_theta_none)

acc = accuracy_score(test_df["label"], test_pred_threshold)
_, _, f1, _ = precision_recall_fscore_support(
    test_df["label"], test_pred_threshold, labels=LABELS_5WAY, average="macro", zero_division=0
)

print(f"\nConfidence-thresholded 5-way (theta_none={best_theta_none}) on test set")
print(f"Accuracy: {acc:.4f}  Macro F1: {f1:.4f}")
print()
print(classification_report(test_df["label"], test_pred_threshold, labels=LABELS_5WAY, digits=4, zero_division=0))

cm = confusion_matrix(test_df["label"], test_pred_threshold, labels=LABELS_5WAY)
pd.DataFrame(
    cm,
    index=[f"true_{l}" for l in LABELS_5WAY],
    columns=[f"pred_{l}" for l in LABELS_5WAY]
)

[Transparency check] Validation macro F1 at theta_none=0.55: 0.6887

Confidence-thresholded 5-way (theta_none=0.55) on test set
Accuracy: 0.6691  Macro F1: 0.6767

                   precision    recall  f1-score   support

             None     0.7281    0.7375    0.7328       541
      beoordeling     0.6259    0.7095    0.6651       389
       beslissing     0.9000    0.7826    0.8372        69
 materiele feiten     0.6395    0.5556    0.5946       297
proceshandelingen     0.5692    0.5388    0.5536       206

         accuracy                         0.6691      1502
        macro avg     0.6925    0.6648    0.6767      1502
     weighted avg     0.6702    0.6691    0.6681      1502



,pred_None,pred_beoordeling,pred_beslissing,pred_materiele feiten,pred_proceshandelingen
true_None,399,78,4,35,25
true_beoordeling,55,276,2,31,25
true_beslissing,11,2,54,2,0
true_materiele feiten,67,31,0,165,34
true_proceshandelingen,16,54,0,25,111
